# 5 — Scoring and transfer

**Needs SCimilarity `model_v1.1`, a GPU, and two atlases. Runs in a couple of minutes.**

Selection answers "which genes separate this population from its reference". Scoring asks the
other question: **given a panel, how much of that program does each individual cell carry?**

The two share one definition of the contrast, which is what makes it meaningful to select on one
dataset and score on another. Here we do exactly that — a panel selected on the Domínguez Conde
immune atlas, frozen, and applied to cytotoxic T cells from a completely different study
(Stephenson et al.) with different donors.

The notebook ends with the part that is easy to get wrong: **what actually transfers, and what
quietly gets recomputed from the data you are scoring.**

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import recast
from recast.encoders import SCimilarityEncoder
from scimilarity.utils import align_dataset

MODEL = "/data/gli9/Jian/sc_age_clock/models/scimilarity_model/model_v1.1"
DATA  = "/data/gli9/test_sig/scattr_benchmark/phase2"
CYTO  = ["Tem/Temra cytotoxic T cells", "Tcm/Naive cytotoxic T cells",
         "Tem/Trm cytotoxic T cells", "gdT"]


go  = pd.read_csv(f"{MODEL}/gene_order.tsv", header=None)[0].tolist()
enc = SCimilarityEncoder(MODEL, device="cuda", normalize=True)

def cytotoxic_lineage(name):
    A = ad.read_h5ad(f"{DATA}/{name}_fullgene.h5ad")
    A = A[A.obs["label"].isin(CYTO)].copy()
    A.obs["label"] = A.obs["label"].cat.remove_unused_categories()
    return align_dataset(A, go)

src = cytotoxic_lineage("dominguez_conde")   # select here
tgt = cytotoxic_lineage("stephenson")        # score here

print("source:", src.shape, " target:", tgt.shape)
print("identical gene axis:", list(src.var_names) == list(tgt.var_names))
print("\ntarget composition:\n" + tgt.obs["label"].value_counts().to_string())

source: (3750, 28231)  target: (5596, 28231)
identical gene axis: True

target composition:
label
Tem/Temra cytotoxic T cells    2172
Tcm/Naive cytotoxic T cells    1387
Tem/Trm cytotoxic T cells      1259
gdT                             778


Both objects were aligned to the same encoder gene order, so their gene axes are identical
element for element. That is what makes a panel portable between them.

## Step 1 — select, on the source dataset only

`cluster_attribution` is `attribute` with the settings the manuscript's scoring path uses
(`baseline="reference"`), run for every label at once. Because the object is already subset to
the cytotoxic lineage, `reference="rest"` *is* the sibling contrast.

In [2]:
res_src = recast.cluster_attribution(enc, src, "label", reference="rest",
                                    device="cuda", qc="silent")
panels = {c: res_src.top(c, 50) for c in CYTO}      # freeze the top 50 per subtype

for c in CYTO:
    print(f"{c:32s} {panels[c][:6]} ...")

Tem/Temra cytotoxic T cells      ['GZMH', 'GZMB', 'GNLY', 'NKG7', 'KLRD1', 'CST7'] ...
Tcm/Naive cytotoxic T cells      ['CCR7', 'NOSIP', 'IL7R', 'LEF1', 'RCAN3', 'TRABD2A'] ...
Tem/Trm cytotoxic T cells        ['GZMK', 'CMC1', 'DUSP2', 'COTL1', 'CD74', 'CD27'] ...
gdT                              ['KLRB1', 'TYROBP', 'TRDC', 'TRDV2', 'TRGC1', 'KLRC1'] ...


The target dataset has now been read but has taken no part in producing those panels.

## Step 2 — score every target cell

`score_cells_attribution_weighted_expression` gives, for each cell and each candidate subtype,

$$S_i(c)\;=\;\operatorname{mean}_{g \in G_c}\;\max(0,\; x_{ig}-C_{\mathrm{ref},g}[c])\;\cdot\;\max(0,\;\varphi_c[g])$$

— each panel gene's reference-relative over-expression in that cell, weighted by how much the
gene drove the encoder's contrast. Passing `_result=res_src` reuses the **source** attribution, so
both the gene panel and the weights $\varphi$ are frozen; `calibrate="zscore"` is a label-free
per-state rescale that makes the columns comparable for an argmax.

In [3]:
S = recast.score_cells_attribution_weighted_expression(
        enc, tgt, "label", panels, reference="rest", device="cuda",
        calibrate="zscore", _result=res_src)

y, pred = tgt.obs["label"].astype(str).values, S.idxmax(axis=1).values
print("per-subtype one-vs-rest AUROC:")
for c in CYTO:
    print(f"  {c:32s} {roc_auc_score((y == c).astype(int), S[c].values):.3f}")
print(f"\nbalanced accuracy of the argmax label: {balanced_accuracy_score(y, pred):.3f}")
print("\nconfusion (rows = true label):")
print(pd.crosstab(pd.Series(y, name="true"), pd.Series(pred, name="pred")).to_string())

per-subtype one-vs-rest AUROC:
  Tem/Temra cytotoxic T cells      0.918
  Tcm/Naive cytotoxic T cells      0.982
  Tem/Trm cytotoxic T cells        0.903
  gdT                              0.974

balanced accuracy of the argmax label: 0.846

confusion (rows = true label):
pred                         Tcm/Naive cytotoxic T cells  Tem/Temra cytotoxic T cells  Tem/Trm cytotoxic T cells  gdT
true                                                                                                                 
Tcm/Naive cytotoxic T cells                         1375                            0                         11    1
Tem/Temra cytotoxic T cells                          106                         1703                        144  219
Tem/Trm cytotoxic T cells                            133                           75                       1006   45
gdT                                                    0                            8                        141  629


A panel estimated on one study separates the same four subtypes in another study, in cells it
never saw, from different donors.

The errors are worth more than the headline number. Naive cells are recovered almost perfectly —
12 of 1,387 misassigned — while roughly seven in ten of all mistakes fall *among* Temra, Trm and
γδ, with Temra→γδ the single largest off-diagonal cell. Those are the three populations
[Tutorial 3](03_choosing_the_reference) measured as sitting closest together, where the
Temra-versus-other-effectors contrast had the lowest `d'` of any question we asked. The hard
contrast at selection time is the hard classification at scoring time — the same fact, seen twice.

## Does freezing cost anything?

The comparison that makes the transfer claim meaningful is against refitting everything on the
target dataset — panels, weights and all.

In [4]:
res_tgt = recast.cluster_attribution(enc, tgt, "label", reference="rest", device="cuda", qc="silent")
S_local = recast.score_cells_attribution_weighted_expression(
        enc, tgt, "label", {c: res_tgt.top(c, 50) for c in CYTO}, reference="rest",
        device="cuda", calibrate="zscore", _result=res_tgt)

cmp = pd.DataFrame({
    "transferred from source": [roc_auc_score((y == c).astype(int), S[c].values) for c in CYTO],
    "refit on target":         [roc_auc_score((y == c).astype(int), S_local[c].values) for c in CYTO],
}, index=CYTO).round(3)
cmp.loc["balanced accuracy (argmax)"] = [balanced_accuracy_score(y, pred),
                                         balanced_accuracy_score(y, S_local.idxmax(axis=1).values)]
cmp

,transferred from source,refit on target
Tem/Temra cytotoxic T cells,0.918000,0.925000
Tcm/Naive cytotoxic T cells,0.982000,0.988000
Tem/Trm cytotoxic T cells,0.903000,0.911000
gdT,0.974000,0.996000
balanced accuracy (argmax),0.845737,0.896716


Refitting on the target wins, and it should — it gets to see the cells it is scoring. But the
margin is small: a fraction of a point of AUROC per subtype, and about five points of balanced
accuracy, most of it on γδ. What crosses the study boundary is most of what there was to have.

:::{important}
**The leakage rule, stated plainly.**

Three ingredients go into the score above. Freezing the first two is what "transfer" means here;
the third is computed on whatever object you hand the scorer, and that is the part people
misread.

| ingredient | in this notebook | needs target labels? |
|---|---|---|
| the gene panel `G_c` | frozen from the source | no |
| the attribution weights `φ_c` | frozen from the source (`_result=res_src`) | no |
| the reference centroid `C_ref[c]` | **recomputed on the target** | **yes** |

`C_ref` is the mean profile of subtype *c*'s reference cells, and `reference="rest"` resolves that
set using the target object's own `cluster_key`. So this measures whether a *program* transfers
across studies. It is not a blind classifier for an unannotated dataset, and reporting it as one
would be overclaiming.

Separately, the default call — without `_result` — fits the attribution on **all** the cells it is
about to score. That is transductive, and it is the right thing when you are inspecting or
labelling within one annotated object. It is the wrong thing for an unbiased supervised benchmark,
where the attribution must be fit on a training split and applied to held-out cells. The library's
docstring says this too; it is repeated here because it is the single easiest way to publish an
optimistic number by accident.
:::

## Where to go next

You have now seen the whole loop: pick a contrast, check it is real, select a panel, and reuse it
somewhere else. What is left is reference material rather than narrative.

- [Usage reference](../usage) — every argument, the composite modes, and the CLI.
- `reproduce/` in the repository — the manuscript's numbers regenerated end to end.